# Transformasi geometri
Transformasi geometri adalah proses memindahkan atau mengubah bentuk dan ukuran suatu objek dalam bidang matematika. Secara sederhana, kamu bisa membayangkan objek tersebut "bermain" dengan koordinatnya, mulai dari digeser posisinya (translasi), dicerminkan agar berbalik arah (refleksi), diputar pada sudut tertentu (rotasi), hingga diperbesar atau diperkecil ukurannya (dilatasi). Konsep ini sangat berguna untuk memetakan perubahan posisi suatu titik atau garis ke posisi baru menggunakan rumus tertentu, sehingga sering digunakan dalam berbagai bidang seperti desain grafis, arsitektur, hingga pembuatan animasi.

contol soal:

In [ ]:
from IPython.display import IFrame

# Link embed dari GeoGebra yang kamu berikan
geogebra_url = "https://www.geogebra.org/calculator/hubc3bhq?embed"

# Menampilkan GeoGebra dengan ukuran yang nyaman
IFrame(src=geogebra_url, width=1000, height=600)

Jawab:

Simulasi Interaktif Geometri: Transformasi Koordinat
Dokumentasi ini menjelaskan implementasi kodingan Python menggunakan Matplotlib untuk memvisualisasikan pergerakan dua objek poligon yang bergerak secara vertikal. Simulasi ini dirancang agar menyerupai antarmuka GeoGebra.
1. Tujuan Simulasi
Simulasi ini bertujuan untuk menunjukkan visualisasi real-time perpindahan titik koordinat dalam sistem Kartesius. Fitur utama meliputi:
Label Koordinat Otomatis: Menampilkan nama titik beserta posisi
(
x
,
y
)
(x,y) yang berubah sesuai pergerakan.
Kontrol Video: Fitur Play/Pause/Slider melalui output HTML5 di Google Colab.
2. Penjelasan Teknis Kodingan
Setup GeoGebra Style
Sistem mengatur batas sumbu agar menyerupai tampilan GeoGebra:
Sumbu x: -1 sampai 7
Sumbu y: -6 sampai 6
Grid: Penggunaan minorticks_on() berfungsi untuk membuat grid halus agar pembacaan titik lebih akurat.
Data Koordinat & Logika Gerak
Penyimpanan Data: Menyimpan posisi awal (start) dan target (target).
Pola Gerak: Kotak Biru diatur bergerak turun, sedangkan Kotak Merah bergerak naik.
Transisi Halus: Pergerakan diatur oleh variabel t yang menggunakan fungsi np.sin untuk menghasilkan transisi mulus dan berkelanjutan dari nilai 0 ke 1.
Logika Label Dinamis: Menggunakan kondisi if t < 0.5. Nama titik akan berubah secara otomatis (contoh: dari huruf A menjadi K) tepat saat kotak melewati titik tengah perjalanannya.
3. Komponen Visual
Poligon (Fill): Dibangun menggunakan perintah ax.fill. Nilai alpha=0.15 membuat warna di dalam kotak menjadi transparan sehingga garis grid di belakangnya tetap terlihat jelas.
Scatter Plot: Setiap titik sudut diberikan bingkai putih (edgecolors='white') agar terlihat lebih menonjol.
Text Label: Semua teks label diformat menggunakan f-string agar angka desimal pada koordinat sumbu Y dapat ditampilkan secara presisi.
4. Rumus Matematika Pergerakan
Posisi setiap titik pada setiap frame dihitung dengan rumus interpolasi linear sebagai berikut:
$$y_{pos} = y_{awal} + (y_{target} - y_{awal}) \times t$$
Rumus ini memastikan bahwa setiap sudut kotak bergerak secara sinkron dan bersamaan, sehingga bentuk persegi panjang tetap terjaga sempurna dan tidak berubah (deformasi) selama animasi berlangsung.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.widgets import Button

# 1. Setup Figure & Axis (Gaya GeoGebra)
fig, ax = plt.subplots(figsize=(10, 8)) # Dibuat agak lebar untuk ruang teks koordinat
plt.subplots_adjust(bottom=0.2)

def setup_geogebra_style():
    ax.set_xlim(-1, 7)
    ax.set_ylim(-6, 6)
    ax.axhline(0, color='black', lw=1.5)
    ax.axvline(0, color='black', lw=1.5)
    ax.grid(True, which='major', linestyle='-', color='#d3d3d3', alpha=0.8)
    ax.grid(True, which='minor', linestyle=':', color='#e0e0e0', alpha=0.5)
    ax.minorticks_on()
    ax.set_aspect('equal')

setup_geogebra_style()

# 2. Data Koordinat (Titik-titik sudut)
# Kotak Atas: A, D, C, B -> Target: K, L, J, I
top_x = np.array([2, 3, 3, 2])
top_y_start = np.array([3, 3, 4, 4])
top_y_target = np.array([1, 1, 2, 2])
labels_top_start = ['A', 'D', 'C', 'B']
labels_top_end = ['K', 'L', 'J', 'I']

# Kotak Bawah: E, F, H, G -> Target: M, N, P, O
bot_x = np.array([2, 3, 3, 2])
bot_y_start = np.array([-3, -3, -4, -4])
bot_y_target = np.array([-1, -1, -2, -2])
labels_bot_start = ['E', 'F', 'H', 'G']
labels_bot_end = ['M', 'N', 'P', 'O']

# 3. Objek Grafis
idx = [0, 1, 2, 3, 0]
line_top, = ax.plot([], [], color='blue', lw=2.5, zorder=4)
line_bot, = ax.plot([], [], color='red', lw=2.5, zorder=4)
fill_top = ax.fill([], [], color='blue', alpha=0.15, zorder=2)[0]
fill_bot = ax.fill([], [], color='red', alpha=0.15, zorder=2)[0]
scat_top = ax.scatter([], [], color='blue', edgecolors='white', s=80, zorder=5)
scat_bot = ax.scatter([], [], color='red', edgecolors='white', s=80, zorder=5)

# Label Teks (Dibuat kosong dulu)
texts_top = [ax.text(0, 0, '', color='blue', fontweight='bold', fontsize=10) for _ in range(4)]
texts_bot = [ax.text(0, 0, '', color='red', fontweight='bold', fontsize=10) for _ in range(4)]

is_paused = False

def update(frame):
    if is_paused:
        return [line_top, line_bot, fill_top, fill_bot, scat_top, scat_bot] + texts_top + texts_bot

    # Animasi smooth (0 ke 1)
    t = (np.sin(frame * 0.05) + 1) / 2

    # --- UPDATE KOTAK ATAS ---
    curr_top_y = top_y_start + (top_y_target - top_y_start) * t
    line_top.set_data(top_x[idx], curr_top_y[idx])
    fill_top.set_xy(np.c_[top_x, curr_top_y])
    scat_top.set_offsets(np.c_[top_x, curr_top_y])

    for i, txt in enumerate(texts_top):
        lbl_letter = labels_top_start[i] if t < 0.5 else labels_top_end[i]
        # Format teks: Huruf(X, Y)
        txt.set_text(f"{lbl_letter}({top_x[i]}, {curr_top_y[i]:.1f})")
        txt.set_position((top_x[i] + 0.15, curr_top_y[i] + 0.15))

    # --- UPDATE KOTAK BAWAH ---
    curr_bot_y = bot_y_start + (bot_y_target - bot_y_start) * t
    line_bot.set_data(bot_x[idx], curr_bot_y[idx])
    fill_bot.set_xy(np.c_[bot_x, curr_bot_y])
    scat_bot.set_offsets(np.c_[bot_x, curr_bot_y])

    for i, txt in enumerate(texts_bot):
        lbl_letter = labels_bot_start[i] if t < 0.5 else labels_bot_end[i]
        # Format teks: Huruf(X, Y)
        txt.set_text(f"{lbl_letter}({bot_x[i]}, {curr_bot_y[i]:.1f})")
        txt.set_position((bot_x[i] + 0.15, curr_bot_y[i] - 0.45))

    return [line_top, line_bot, fill_top, fill_bot, scat_top, scat_bot] + texts_top + texts_bot

ani = FuncAnimation(fig, update, frames=200, interval=30, blit=True)

plt.title("Simulasi GeoGebra: Koordinat Dinamis Huruf(X, Y)", pad=20)
ani = FuncAnimation(fig, update, frames=200, interval=30)

from IPython.display import HTML
HTML(ani.to_jshtml())